# Topic 3: The Chat LLM (the generative model)


### The contrast that unlocks everything

You already met the embedding model. It does one job -- turn text into a **vector** so you can measure similarity. It cannot write.

Now meet a **chat LLM** (`llama3:8b`): it reads **text** and **writes new text**. Retrieval and generation are opposite halves of RAG, and they are *different models with different output types*.

| | Embedding (`nomic-embed-text`) | Chat LLM (`llama3:8b`) |
|---|---|---|
| Input | text | text (a prompt) |
| Output | vector (list of floats) | text (an `AIMessage`) |
| Job | measure/find meaning | generate/answer |
| Where in RAG | builds the vectorstore + retrieval | writes the final answer |

### `invoke()` returns an `AIMessage` -- not a string

People expect `invoke()` to return a string and instead get a message object. The actual answer text lives in `.content`.

### Anatomy of an `AIMessage`

- `.content`             -> the answer text (you use this 95% of the time)
- `.response_metadata`   -> model name, timing, eval stats
- `.usage_metadata`      -> input/output/total token counts (tracking cost)
- `.tool_calls`          -> empty on plain calls -- but this is where the model will *request tool calls* later (the seed of the **Agents** topic)

### Getting the text out: two ways

1. **Manually** -- `msg.content`
2. **Automatically** -- pipe through `StrOutputParser()`, which unwraps `.content` for you in a chain

Plus **streaming** -- tokens arrive bit by bit (the chat "typing" effect):

```python
for chunk in llm.stream("prompt"):
    print(chunk.content, end="")
```

> Live observation: asked a general-knowledge question with no context, the model confidently *fabricated* a wrong answer ("CycleGAN by Jeremy J. Wu") -- smooth but wrong. That is a hallucination on screen, and it is exactly why RAG feeds the model context and why the Topic 2 grounding guard exists.

## Topic 3 complete -- the summary

- Embeddings -> **vectors** (measure meaning); chat LLM -> **text** (generate answers). Opposite jobs, different models.
- `ChatOllama(model="llama3:8b").invoke(prompt)` returns an **`AIMessage`**, not a string -- the answer is in **`.content`**.
- `AIMessage` also carries `.response_metadata` (model, timing, tokens) and `.usage_metadata` (input/output/total tokens), and `.tool_calls` -- the future hook for Agents.
- Two ways to the text: `.content` manually, or `StrOutputParser()` to unwrap automatically in a chain (v1.x returns a `TextAccessor`, a `str` subclass -- behaves like a normal string).
- **Streaming**: `llm.stream(...)` yields `.content` chunks token-by-token -- the typing effect in chat UIs.
- Live proof of hallucination: it fabricated a wrong paper without context, and got RAG right with it -- grounding decides the difference.

In [ ]:
# ---- The Chat LLM: embed vs generate ----
# What does invoke() RETURN? Not a bare string -- a wrapped object.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:8b")

result = llm.invoke("In one sentence, what is retrieval augmented generation?")
print("return type:", type(result).__name__)
print("the .content (the actual answer):")
print(result.content)

In [ ]:
# ---- Anatomy of an AIMessage ----
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:8b")
msg = llm.invoke("Say exactly: hello")

print(".content           :", msg.content)                    # the answer text
print(".response_metadata :", msg.response_metadata.get("model"), "|", msg.response_metadata.get("eval_count"), "tokens")
print(".usage_metadata    :", dict(msg.usage_metadata or {})) # token counts
print(".tool_calls        :", msg.tool_calls)                 # empty now -- foreshadows Agents

In [ ]:
# ---- Manual .content vs StrOutputParser + streaming ----
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:8b")
parser = StrOutputParser()

# manual
msg = llm.invoke("Name the most famous Transformer paper. One line.")
print("manual .content:", msg.content)

# through the parser (str subclass in v1.x -- fully usable as a string)
out = parser.invoke(msg)
print("parser.invoke(msg) ->", type(out).__name__, "| isinstance str:", isinstance(out, str))

# streaming -- the "typing" effect
print("\nstreaming tokens:")
for chunk in llm.stream("Count from 1 to 3."):
    print(repr(chunk.content), end=" ")
print()